In [1]:
import astropy.units as u
import pandas as pd
import logging
import re

In [2]:
import scopesim as sim

In [ ]:
# NB: should pull from scopesim and irdb githubs together
sim.rc.__config__["!SIM.file.local_packages_path"] = "/home/jessicap/Desktop/Packages/irdb"

In [4]:
# From what I can tell (which is very little at the moment) the way to 
# record the calculated DIT if it is under MINDIT, is to pull it from the 
# stdout warning that scopesim prints

# This is just setting up the system to record the output of the warning
class DebugCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.raw = []

    def emit(self, record):
        self.raw.append(record.getMessage())

debug_capture = DebugCapture()

logger = logging.getLogger('astar.scopesim.effects.electronic')
print("effective level:", logger.getEffectiveLevel())  # should be <= WARNING (30)
logger.setLevel(logging.WARNING)          # force it open regardless of parent config
logger.addHandler(debug_capture)          # attach directly, don't rely on propagation
logger.propagate = True

effective level: 10


In [6]:
results = []

for mode in ['high_capacity', 'low_capacity']:
    cmd = sim.UserCommands(use_instrument="METIS", set_modes=[f'wcu_img_n'],
                           properties={'!OBS.detector_readout_mode': mode})
    cmd['!OBS.auto_exposure.fill_frac'] = 0.75

    opt_train = sim.OpticalTrain(cmd)

    nd_filters = list(opt_train['nd_filter_wheel'].filters.keys())
    nd_filters.remove('closed')

    filters = list(opt_train['filter_wheel'].filters.keys())
    filters.remove('closed')
    filters.remove('N_spec')

    for nd_filt in nd_filters:
        opt_train['nd_filter_wheel'].change_filter(nd_filt)
        for filter in filters:
            opt_train['filter_wheel'].change_filter(filter)

            opt_train.observe()

            debug_capture.raw.clear()

            outhdul = opt_train.readout(exptime=3600)[0]

            attempted_dit = None
            mindit = None
            for msg in debug_capture.raw:
                match = re.search(r"DIT\s*=\s*([\d.]+)\s*s\s*<\s*MINDIT\s*=\s*([\d.]+)", msg)
                if match:
                    attempted_dit = float(match.group(1))
                    mindit = float(match.group(2))
                    break

            det_no = 2 
            gain = outhdul[1].header[f"ESO DET{det_no} CHIP GAIN"] * u.electron / u.adu
            full_well = outhdul[1].header[f"ESO DET{det_no} CHIP FULLWELL"] * u.electron
            outimg = outhdul[1].data * u.adu * gain
            fill_frac = outimg.max() / full_well << u.percent
            dit = outhdul[0].header['HIERARCH ESO DET DIT']


            results.append({
                'nd_filter': nd_filt,
                'detmode': mode,
                'filter': filter,
                'dit': dit,
                'fill_frac': fill_frac,
                'mindit_triggered': attempted_dit is not None,
                'attempted_dit': attempted_dit,
                'mindit': mindit,
            })

results_df = pd.DataFrame(results)

results_df.to_csv(f"N_IMG_results.csv")

astar.scopesim.optics.optical_train - Observing empty field


 FOVs:   0%|          | 0/31 [00:00<?, ?it/s]

astar.scopesim.effects.electronic - WARNING: DIT = 0.000 s < MINDIT = 0.011 s
astar.scopesim.effects.electronic - WARNING: The detector will likely be saturated!
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.optics.optical_train - Observing empty field


 FOVs:   0%|          | 0/11 [00:00<?, ?it/s]

astar.scopesim.effects.electronic - WARNING: DIT = 0.002 s < MINDIT = 0.011 s
astar.scopesim.effects.electronic - WARNING: The detector will likely be saturated!
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.optics.optical_train - Observing empty field


...

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.optics.optical_train - Observing empty field


 FOVs:   0%|          | 0/4 [00:00<?, ?it/s]

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.optics.optical_train - Observing empty field


 FOVs:   0%|          | 0/4 [00:00<?, ?it/s]

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
